# 01. 데이터 전처리

팀원 5명이 각자 수집한 데이터를 통합하고, 텍스트 정제 → 중복 제거 → 광고 필터링을 거쳐 최종 분석용 데이터를 생성합니다.

| 단계 | 내용 |
|------|------|
| 데이터 통합 | 팀원별 pkl/csv 파일 병합 |
| 텍스트 정제 | HTML, URL, 이메일, 특수문자, 이모지 제거 |
| 필터링 | 15자 미만 제거, 중복 제거, 광고 제거 |
| 결과 | 약 7만 건 → **36,245건** |

## 1. 팀원별 데이터 통합

팀원 5명이 각자 네이버 블로그, 카페, 지식인 등에서 수집한 데이터를 하나로 합칩니다.

In [48]:
import pandas as pd
import pickle
 
with open('카페_외출_산책등_2만개.pkl', "rb") as f:
    df_재민 = pickle.load(f)

with open('건강모음(필사즉생).pkl', "rb") as f:
    df_진규1 = pickle.load(f)

with open('지식인(필사즉생).pkl', "rb") as f:
    df_진규12 = pickle.load(f)

with open('강아쥐(필사즉생) (1).pkl', "rb") as f:
    df_진규123 = pickle.load(f)



df_지나 = pd.read_csv('소리.csv', encoding='cp949')
df_효재 = pd.read_csv('강아지취합2_5000.csv', encoding='cp949')
df_민철 = pd.read_csv('df_하우스임시.csv', encoding='utf-8-sig')

### 컬럼 통일 후 병합

팀원마다 컬럼 구조가 달라서, 불필요한 컬럼을 제거하고 통일합니다.

In [46]:
# 불필요 컬럼 제거 (팀원별 컬럼 구조 통일)
df_효재 = df_효재.drop(['date'], axis=1)
df_진규123 = df_진규123.drop(['date', 'url', 'nouns', 'comments', '주소', '명사'], axis=1)

# 전체 병합
final_df = pd.concat([df_재민, df_진규1, df_진규12, df_진규123, df_지나, df_효재, df_민철])
final_df = final_df.drop(['date'], axis=1, errors='ignore')

# content 기준 중복 제거
final_df = final_df.drop_duplicates(subset=['content']).reset_index(drop=True)

## 2. 텍스트 정제 (clean_text)

HTML 태그, URL, 이메일, 특수문자, 이모지를 제거하여 분석에 적합한 텍스트로 변환합니다.

In [48]:
final_df=pd.read_csv('df_하우스_final2.xls')
final_df

,title,content
0,강쥐 입양 후 스트레스,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....
1,외부 소음 헛짖음 어떻게 교정하셨는지 궁금해요.,안녕하세요.\n1살 된 포메라니안 여자아이를 키우고 있는 견주입니다.\n어릴 때는 ...
2,구래동 샤오마라 점주입니다(동물학대의심건),안녕하세요 구래동 샤오마라점주입니다\n그제 아침에 저희강아지를 동물학대로신고한다는분...
3,첫 분양! 필수 준비물?? 알려주세요 ㅜㅜ,아기 강아지 분양할려하는데\n필수적으로 준비해야할 것들이 뭐가있을까요?\n울타리 배...
4,로얄 테일즈 글로리아 일체형 스탠다드 베이지 유모차 후기 🐶,안녕하세요~\n로얄 테일즈 유모차 내돈내산 구매후기입니당 !!\n기존에 쓰던 유모차...
...,...,...
9906,6키로 비숑 백팩 추천 부탁 드립니다.,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...
9907,성견 후에도 식분증이 원래 있었는지 아닌지 어떻게 아나요?ㅠㅠ,"오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ..."
9908,우다다 에너지소모엔역시 오프리쉬 트레일!!,"몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가\n자꾸 힐, 앉아, 기다려 이..."
9909,동물 파양 입양 자격검증도없이 잘키워주겠다는 말만 믿고 보내지마시길요!동물범죄전과자...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는\n감정있는 소중히 여겨야하고 가...


In [49]:
import re
import pandas as pd
import emoji

def clean_text(text: str, keep_emoji=False) -> str:
    if pd.isna(text):  # NaN 처리
        return ""

    # 1. HTML 태그 제거
    text = re.sub(r"<.*?>", " ", text)

    # 2. URL 제거
    text = re.sub(r"http\S+|www\S+", " ", text)

    # 3. 이메일 제거
    text = re.sub(r"\S+@\S+\.\S+", " ", text)

    # 4. 특수문자 제거 (한글, 영어, 숫자, 기본 구두점만 허용)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s.,!?]", " ", text)

    # 5. 이모지 처리
    if keep_emoji:
        # 이모지를 토큰으로 변환 (예: 😀 → EMO_SMILE)
        text = emoji.demojize(text, delimiters=(" ", " "))
    else:
        # 이모지 제거
        text = emoji.replace_emoji(text, replace=" ")

    # 6. 연속 공백을 하나로 통일
    text = re.sub(r"\s+", " ", text).strip()
        
    return text


final_df["clean_content"] = final_df["content"].apply(lambda x: clean_text(x, keep_emoji=False))

# 각 리뷰 텍스트 길이 계산
final_df["length"] = final_df["clean_content"].str.len()

 #길이가 200 초과인 리뷰만 필터링
df_filtered = final_df[final_df["clean_content"].str.len() > 15].reset_index(drop=True)
df_filtered.head()

,title,content,clean_content,length
0,강쥐 입양 후 스트레스,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,514
1,외부 소음 헛짖음 어떻게 교정하셨는지 궁금해요.,안녕하세요.\n1살 된 포메라니안 여자아이를 키우고 있는 견주입니다.\n어릴 때는 ...,안녕하세요. 1살 된 포메라니안 여자아이를 키우고 있는 견주입니다. 어릴 때는 정말...,965
2,구래동 샤오마라 점주입니다(동물학대의심건),안녕하세요 구래동 샤오마라점주입니다\n그제 아침에 저희강아지를 동물학대로신고한다는분...,안녕하세요 구래동 샤오마라점주입니다 그제 아침에 저희강아지를 동물학대로신고한다는분이...,2068
3,첫 분양! 필수 준비물?? 알려주세요 ㅜㅜ,아기 강아지 분양할려하는데\n필수적으로 준비해야할 것들이 뭐가있을까요?\n울타리 배...,아기 강아지 분양할려하는데 필수적으로 준비해야할 것들이 뭐가있을까요? 울타리 배변패...,73
4,로얄 테일즈 글로리아 일체형 스탠다드 베이지 유모차 후기 🐶,안녕하세요~\n로얄 테일즈 유모차 내돈내산 구매후기입니당 !!\n기존에 쓰던 유모차...,안녕하세요 로얄 테일즈 유모차 내돈내산 구매후기입니당 !! 기존에 쓰던 유모차가 망...,1221


In [69]:
df_filtered[df_filtered['clean_content'].duplicated(keep=False)]

,title,content,clean_content,length
290,실내견으로 완벽 변신한 순둥이가 평생 가족을 찾습니다!!(개st하우스 출연 강쥐),"폭우주의보가 내린 7월, 침수위기에서 구조된 순둥이가 평생 가족을 찾습니다!\n임보...","폭우주의보가 내린 7월, 침수위기에서 구조된 순둥이가 평생 가족을 찾습니다! 임보 ...",904
291,개st하우스 출연 강아지 순둥이가 가족 만날 준비를 마쳤어요!,"폭우주의보가 내린 7월, 침수위기에서 구조된 순둥이가 평생 가족을 찾습니다!\n임보...","폭우주의보가 내린 7월, 침수위기에서 구조된 순둥이가 평생 가족을 찾습니다! 임보 ...",904
807,펫트리움 강아지 캠피체어 체험단 응모하세요~,https://shopping.naver.com/window/brand-pet/st...,네이버쇼핑 네이버펫 당일배송부터 공식 브랜드 혜택까지 shopping.naver.c...,304
808,펫트리움 강아지 캠피체어 무료체험단 모집,https://shopping.naver.com/window/brand-pet/st...,네이버쇼핑 네이버펫 당일배송부터 공식 브랜드 혜택까지 shopping.naver.c...,304


In [70]:
df_filtered = df_filtered.drop_duplicates(subset='clean_content')

In [71]:
df_filtered

,title,content,clean_content,length
0,강쥐 입양 후 스트레스,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,514
1,외부 소음 헛짖음 어떻게 교정하셨는지 궁금해요.,안녕하세요.\n1살 된 포메라니안 여자아이를 키우고 있는 견주입니다.\n어릴 때는 ...,안녕하세요. 1살 된 포메라니안 여자아이를 키우고 있는 견주입니다. 어릴 때는 정말...,965
2,구래동 샤오마라 점주입니다(동물학대의심건),안녕하세요 구래동 샤오마라점주입니다\n그제 아침에 저희강아지를 동물학대로신고한다는분...,안녕하세요 구래동 샤오마라점주입니다 그제 아침에 저희강아지를 동물학대로신고한다는분이...,2068
3,첫 분양! 필수 준비물?? 알려주세요 ㅜㅜ,아기 강아지 분양할려하는데\n필수적으로 준비해야할 것들이 뭐가있을까요?\n울타리 배...,아기 강아지 분양할려하는데 필수적으로 준비해야할 것들이 뭐가있을까요? 울타리 배변패...,73
4,로얄 테일즈 글로리아 일체형 스탠다드 베이지 유모차 후기 🐶,안녕하세요~\n로얄 테일즈 유모차 내돈내산 구매후기입니당 !!\n기존에 쓰던 유모차...,안녕하세요 로얄 테일즈 유모차 내돈내산 구매후기입니당 !! 기존에 쓰던 유모차가 망...,1221
...,...,...,...,...
9826,6키로 비숑 백팩 추천 부탁 드립니다.,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,141
9827,성견 후에도 식분증이 원래 있었는지 아닌지 어떻게 아나요?ㅠㅠ,"오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...","오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...",358
9828,우다다 에너지소모엔역시 오프리쉬 트레일!!,"몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가\n자꾸 힐, 앉아, 기다려 이...","몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가 자꾸 힐, 앉아, 기다려 이런...",396
9829,동물 파양 입양 자격검증도없이 잘키워주겠다는 말만 믿고 보내지마시길요!동물범죄전과자...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는\n감정있는 소중히 여겨야하고 가...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는 감정있는 소중히 여겨야하고 가족...,3084


## 3. 필터링

### 3-1. 짧은 리뷰 제거

15자 미만의 리뷰는 의미 있는 분석이 어려우므로 제거합니다.

In [22]:
df_filtered = final_df[final_df["clean_content"].str.len() > 15].reset_index(drop=True)
df_filtered.head()

,title,content,clean_content
0,강쥐 입양 후 스트레스,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....
1,외부 소음 헛짖음 어떻게 교정하셨는지 궁금해요.,안녕하세요.\n1살 된 포메라니안 여자아이를 키우고 있는 견주입니다.\n어릴 때는 ...,안녕하세요. 1살 된 포메라니안 여자아이를 키우고 있는 견주입니다. 어릴 때는 정말...
2,구래동 샤오마라 점주입니다(동물학대의심건),안녕하세요 구래동 샤오마라점주입니다\n그제 아침에 저희강아지를 동물학대로신고한다는분...,안녕하세요 구래동 샤오마라점주입니다 그제 아침에 저희강아지를 동물학대로신고한다는분이...
3,첫 분양! 필수 준비물?? 알려주세요 ㅜㅜ,아기 강아지 분양할려하는데\n필수적으로 준비해야할 것들이 뭐가있을까요?\n울타리 배...,아기 강아지 분양할려하는데 필수적으로 준비해야할 것들이 뭐가있을까요? 울타리 배변패...
4,로얄 테일즈 글로리아 일체형 스탠다드 베이지 유모차 후기 🐶,안녕하세요~\n로얄 테일즈 유모차 내돈내산 구매후기입니당 !!\n기존에 쓰던 유모차...,안녕하세요 로얄 테일즈 유모차 내돈내산 구매후기입니당 !! 기존에 쓰던 유모차가 망...


In [72]:
len(df_filtered)

9829

### 3-2. 광고 제거

In [73]:
import re
import pandas as pd

def is_advertisement(text: str) -> bool:
    if pd.isna(text):
        return False

    # 1) URL 포함
    if re.search(r"http\S+|www\S+", text):
        return True

    # 2) 이메일/전화번호 포함
    if re.search(r"\S+@\S+\.\S+", text) or re.search(r"\d{8,}", text):
        return True

    # 3) 카톡/상담/구매 관련 키워드
    ad_keywords = ["카톡", "상담", "구매문의", "최저가", "특가", "할인", "문의주세요", "클릭"]
    if any(keyword in text for keyword in ad_keywords):
        return True

    return False

In [74]:
# 광고 여부 컬럼 추가
df_filtered["is_ad"] = df_filtered["content"].apply(is_advertisement)

# 광고 리뷰 제거
df_filtered = df_filtered[~df_filtered["is_ad"]].reset_index(drop=True)

df_filtered

C:\Users\rhals\AppData\Local\Temp\ipykernel_10316\1621300364.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["is_ad"] = df_filtered["content"].apply(is_advertisement)


,title,content,clean_content,length,is_ad
0,강쥐 입양 후 스트레스,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,이제 온지 3일차된 보호소 출신 3개월 강쥐때문에 너무 신경쓰여서 스트레스 받아요....,514,False
1,외부 소음 헛짖음 어떻게 교정하셨는지 궁금해요.,안녕하세요.\n1살 된 포메라니안 여자아이를 키우고 있는 견주입니다.\n어릴 때는 ...,안녕하세요. 1살 된 포메라니안 여자아이를 키우고 있는 견주입니다. 어릴 때는 정말...,965,False
2,첫 분양! 필수 준비물?? 알려주세요 ㅜㅜ,아기 강아지 분양할려하는데\n필수적으로 준비해야할 것들이 뭐가있을까요?\n울타리 배...,아기 강아지 분양할려하는데 필수적으로 준비해야할 것들이 뭐가있을까요? 울타리 배변패...,73,False
3,펫샵과 신종펫샵이 안망하는 이유(앞담화),강아지 키우려는 사람들이 준비 없이 키워서 그래요.\n한 생명 15년 이상 키우는데...,"강아지 키우려는 사람들이 준비 없이 키워서 그래요. 한 생명 15년 이상 키우는데,...",684,False
4,또 궁금한게 생겼어요!,푸들/3살/5키로/ 중성화X/수컷\n저희집 온지 16일째네요😀\n저희는 울타리 생활...,푸들 3살 5키로 중성화X 수컷 저희집 온지 16일째네요 저희는 울타리 생활중이예요...,235,False
...,...,...,...,...,...
9149,6키로 비숑 백팩 추천 부탁 드립니다.,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,141,False
9150,성견 후에도 식분증이 원래 있었는지 아닌지 어떻게 아나요?ㅠㅠ,"오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...","오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...",358,False
9151,우다다 에너지소모엔역시 오프리쉬 트레일!!,"몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가\n자꾸 힐, 앉아, 기다려 이...","몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가 자꾸 힐, 앉아, 기다려 이런...",396,False
9152,동물 파양 입양 자격검증도없이 잘키워주겠다는 말만 믿고 보내지마시길요!동물범죄전과자...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는\n감정있는 소중히 여겨야하고 가...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는 감정있는 소중히 여겨야하고 가족...,3084,False


In [62]:
df_filtered.to_csv('강쥐(필사즉생)_임시최종.csv', encoding = 'utf-8-sig', index = False)

## 데이터 추가수집 후 최종 사용 데이터 

In [77]:
df_임시최종=pd.read_csv('강쥐(필사즉생)_임시최종.xls')

In [78]:
df_임시최종2=pd.concat([df_임시최종,df_filtered],axis=0, ignore_index=True)

In [81]:
df_임시최종2

,title,content,clean_content,is_ad,length
0,산책중에 개가 개 보고 사납게 짖는거,봉구를 보자마자 사납게 짖던데요 갈색 푸들이....\n봉구는 내가 뭘 이러고 있고 ...,봉구를 보자마자 사납게 짖던데요 갈색 푸들이.... 봉구는 내가 뭘 이러고 있고 그...,False,NaN
1,오늘 산책시키면서 깨달았어요.,행복이는 외동으로만 살아야 하나봐요.\n주말만 유모차 태워서 산책시키는데\n거의 산...,행복이는 외동으로만 살아야 하나봐요. 주말만 유모차 태워서 산책시키는데 거의 산책 ...,False,NaN
2,산책시 아무거나 주워먹어 걱정이에요,산책시 땅에 떨어져있는 음식이나 풀 흙등 뭐든 누워먹어서 걱정이에요 ㅜ ㅜ 그러다 ...,산책시 땅에 떨어져있는 음식이나 풀 흙등 뭐든 누워먹어서 걱정이에요 그러다 잘못먹으...,False,NaN
3,서울근교 애견동반펜션 추천해주세요!,7월 중순에 강아지랑 같이 가족여행 갈려고 하는데\n애견동반펜션 좋은곳이랑 근처 동...,7월 중순에 강아지랑 같이 가족여행 갈려고 하는데 애견동반펜션 좋은곳이랑 근처 동반...,False,NaN
4,장마철 산책고민하다 유모차 구입했어요,저희집 강아지가 물을 엄청 무서워하는데 산책은 꼭 해야하는 아이라ㅠ\n고민하다가 유...,저희집 강아지가 물을 엄청 무서워하는데 산책은 꼭 해야하는 아이라 고민하다가 유모차...,False,NaN
...,...,...,...,...,...
71030,6키로 비숑 백팩 추천 부탁 드립니다.,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,안녕하세요! 6키로 비숑 키우고 있는데 옆으로 메는 백은 어깨가 나갈것 같아서 6키...,False,141.0
71031,성견 후에도 식분증이 원래 있었는지 아닌지 어떻게 아나요?ㅠㅠ,"오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...","오늘 아침 6시쯤 대변을 보고, 15분쯤 후에 일찍 일어난 다른 가족이 우연히 똥 ...",False,358.0
71032,우다다 에너지소모엔역시 오프리쉬 트레일!!,"몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가\n자꾸 힐, 앉아, 기다려 이...","몇일 오프리쉬 갔다가 또 몇일 그냥 동네 돌아다니다가 자꾸 힐, 앉아, 기다려 이런...",False,396.0
71033,동물 파양 입양 자격검증도없이 잘키워주겠다는 말만 믿고 보내지마시길요!동물범죄전과자...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는\n감정있는 소중히 여겨야하고 가...,말못하는 동물도 물건 아니고 인간처럼 고통 다느끼는 감정있는 소중히 여겨야하고 가족...,False,3084.0


In [83]:
df_임시최종2=df_임시최종2.drop_duplicates(subset='clean_content')
df_임시최종2.to_csv('df_개인최종.csv',index=False,encoding='utf-8-sig')